# 🌌 EXODUS-SPECULUM - Template Colab v1.0

**Pipeline de Camera Projection Mapping pour transformation vidéo → 3D**

---

## Instructions
1. Exécuter les cellules **dans l'ordre**
2. S'assurer que le **GPU T4** est activé (Runtime → Change runtime type)
3. Autoriser le montage Google Drive quand demandé

---

In [ ]:
# ============================================================
# 🌌 EXODUS-SPECULUM - Template Colab v1.0
# ============================================================
# Ce notebook est le point d'entrée du pipeline.
# Exécuter les cellules dans l'ordre.
# ============================================================

import os
import sys

# Configuration Google Drive
DRIVE_MOUNT = "/content/drive"
SANCTUM_PATH = "/content/drive/MyDrive/EXODUS-SPECULUM"

# Vérification GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ ATTENTION: GPU non disponible! Activer Runtime → Change runtime type → T4 GPU")

In [ ]:
# ============================================================
# 📁 Montage Google Drive
# ============================================================
from google.colab import drive
drive.mount(DRIVE_MOUNT)

# Vérifier structure Sanctum
required_dirs = [
    "00_SOURCE",
    "01_SCANNER_OUT",
    "02_CORTEX_OUT",
    "03_ASSETS_LIBRARY",
    "04_BLENDER_PROJECTS",
    "05_RENDER_FARM",
    "06_FINAL_PRODUCT"
]

print("\n📂 Structure du Sanctum EXODUS-SPECULUM:")
print("=" * 50)

for d in required_dirs:
    path = os.path.join(SANCTUM_PATH, d)
    if not os.path.exists(path):
        os.makedirs(path)
        print(f"✅ Créé: {d}/")
    else:
        print(f"📁 Existant: {d}/")

print("=" * 50)
print(f"✅ Sanctum prêt: {SANCTUM_PATH}")

In [ ]:
# ============================================================
# 📦 Installation Dépendances
# ============================================================
# Cette cellule installe TOUTES les dépendances requises.
# Temps estimé: 3-5 minutes

print("🔧 Installation des dépendances...")
print("=" * 50)

# Core ML (déjà présent sur Colab mais on s'assure de la version CUDA)
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
print("✅ PyTorch + CUDA 11.8")

# Image processing
!pip install -q numpy opencv-python pillow
print("✅ NumPy, OpenCV, Pillow")

# Blender headless (critique)
print("\n🔨 Installation Blender headless (bpy)...")
print("⏳ Cette étape peut prendre 2-3 minutes...")

# Méthode 1: Installation via pip (bpy wheel)
# Note: bpy sur pip peut nécessiter des dépendances système
!apt-get update -qq
!apt-get install -qq -y libxi6 libxxf86vm1 libxfixes3 libxrender1 libgl1
!pip install -q bpy==4.0.0

print("\n" + "=" * 50)
print("🔍 Vérification de l'installation bpy...")

# Vérification immédiate
try:
    import bpy
    print(f"✅ Blender version: {bpy.app.version_string}")
    print("🔨 BPY IMPORT RÉUSSI - BLENDER HEADLESS OPÉRATIONNEL")
except ImportError as e:
    print(f"❌ ÉCHEC import bpy: {e}")
    print("\n🔄 Tentative méthode alternative...")
    # Méthode alternative: installation système
    !apt-get install -qq -y blender
    print("⚠️ Blender installé via apt. Utiliser 'blender --background --python script.py'")

In [ ]:
# ============================================================
# 🔨 Test Blender Headless
# ============================================================
import bpy
import os

print("🔨 Test de Blender Headless avec rendu GPU...")
print("=" * 50)

# Reset scene
bpy.ops.wm.read_factory_settings(use_empty=True)
print("✅ Scene reset")

# Créer un cube de test
bpy.ops.mesh.primitive_cube_add(size=2, location=(0, 0, 0))
cube = bpy.context.active_object
cube.name = "TestCube_SPECULUM"
print(f"✅ Cube créé: {cube.name}")

# Ajouter une caméra
bpy.ops.object.camera_add(location=(5, -5, 5))
camera = bpy.context.active_object
camera.name = "Camera_SPECULUM"
print(f"✅ Caméra créée: {camera.name}")

# Pointer la caméra vers le cube
from mathutils import Vector
direction = Vector((0, 0, 0)) - camera.location
camera.rotation_euler = direction.to_track_quat('-Z', 'Y').to_euler()
print("✅ Caméra orientée vers le cube")

# Ajouter une lumière
bpy.ops.object.light_add(type='SUN', location=(5, 5, 10))
light = bpy.context.active_object
light.name = "Sun_SPECULUM"
print(f"✅ Lumière créée: {light.name}")

# Configuration rendu de test
scene = bpy.context.scene
scene.camera = camera
scene.render.engine = 'CYCLES'
scene.cycles.device = 'GPU'
scene.render.resolution_x = 256
scene.render.resolution_y = 256
scene.cycles.samples = 8
print("✅ Configuration rendu: CYCLES GPU, 256x256, 8 samples")

# Activer GPU
print("\n🔧 Configuration GPU Cycles...")
try:
    prefs = bpy.context.preferences.addons['cycles'].preferences
    prefs.compute_device_type = 'CUDA'
    prefs.get_devices()
    
    gpu_found = False
    for device in prefs.devices:
        device.use = True
        print(f"  ✅ Device activé: {device.name} ({device.type})")
        if device.type == 'CUDA':
            gpu_found = True
    
    if gpu_found:
        print("✅ GPU CUDA détecté et activé")
    else:
        print("⚠️ Aucun GPU CUDA détecté, rendu CPU")
        
except Exception as e:
    print(f"⚠️ Configuration GPU: {e}")
    print("   Fallback sur rendu CPU")

# Rendu test
print("\n🎬 Lancement du rendu test...")
output_path = "/content/test_render_speculum.png"
scene.render.filepath = output_path

import time
start_time = time.time()
bpy.ops.render.render(write_still=True)
render_time = time.time() - start_time

# Vérification
print("\n" + "=" * 50)
if os.path.exists(output_path):
    file_size = os.path.getsize(output_path) / 1024
    print(f"✅ SUCCÈS: Rendu test sauvegardé")
    print(f"   📍 Chemin: {output_path}")
    print(f"   📐 Résolution: 256x256")
    print(f"   📦 Taille: {file_size:.1f} KB")
    print(f"   ⏱️ Temps: {render_time:.2f}s")
    print("\n🔨 BLENDER HEADLESS OPÉRATIONNEL")
else:
    print("❌ ÉCHEC: Le rendu n'a pas été créé")
    print("   Vérifier les logs ci-dessus pour les erreurs")

In [ ]:
# ============================================================
# 🖼️ Affichage du rendu test
# ============================================================
from PIL import Image
import matplotlib.pyplot as plt

output_path = "/content/test_render_speculum.png"

if os.path.exists(output_path):
    img = Image.open(output_path)
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.title("Test Render SPECULUM - Blender Cycles GPU")
    plt.axis('off')
    plt.show()
    print("✅ Rendu affiché avec succès")
else:
    print("❌ Fichier de rendu non trouvé")

In [ ]:
# ============================================================
# 📊 Rapport Système SPECULUM
# ============================================================
import subprocess
import sys
import torch
import bpy
import os

print("=" * 60)
print("          RAPPORT SYSTÈME EXODUS-SPECULUM")
print("=" * 60)

# Python
print(f"\n🐍 Python: {sys.version.split()[0]}")

# PyTorch & CUDA
print(f"\n🔥 PyTorch: {torch.__version__}")
print(f"   CUDA compilé: {torch.version.cuda}")
print(f"   CUDA disponible: {torch.cuda.is_available()}")

# Blender
print(f"\n🔨 Blender: {bpy.app.version_string}")

# GPU Details
print("\n🎮 GPU:")
if torch.cuda.is_available():
    print(f"   Nom: {torch.cuda.get_device_name(0)}")
    mem_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    mem_reserved = torch.cuda.memory_reserved(0) / 1e9
    mem_allocated = torch.cuda.memory_allocated(0) / 1e9
    print(f"   VRAM Total: {mem_total:.1f} GB")
    print(f"   VRAM Réservée: {mem_reserved:.2f} GB")
    print(f"   VRAM Allouée: {mem_allocated:.2f} GB")
    print(f"   VRAM Libre: {mem_total - mem_reserved:.2f} GB")
else:
    print("   ⚠️ Aucun GPU détecté")

# Drive & Sanctum
print("\n📁 Storage:")
print(f"   Drive monté: {'✅' if os.path.exists(DRIVE_MOUNT) else '❌'}")
print(f"   Sanctum présent: {'✅' if os.path.exists(SANCTUM_PATH) else '❌'}")

# Validation Checklist
print("\n" + "=" * 60)
print("          CHECKLIST VALIDATION P0")
print("=" * 60)

checks = [
    ("Python 3.10+", sys.version_info >= (3, 10)),
    ("PyTorch installé", 'torch' in sys.modules),
    ("CUDA disponible", torch.cuda.is_available()),
    ("Blender bpy importé", 'bpy' in sys.modules),
    ("Google Drive monté", os.path.exists(DRIVE_MOUNT)),
    ("Sanctum structure", os.path.exists(SANCTUM_PATH)),
    ("Rendu test créé", os.path.exists("/content/test_render_speculum.png")),
]

all_passed = True
for check_name, check_result in checks:
    status = "✅" if check_result else "❌"
    print(f"   {status} {check_name}")
    if not check_result:
        all_passed = False

print("\n" + "=" * 60)
if all_passed:
    print("   🚀 SYSTÈME PRÊT POUR EXODUS-SPECULUM")
else:
    print("   ⚠️ CERTAINS CHECKS ONT ÉCHOUÉ")
print("=" * 60)

---

## 🚀 Prochaines Étapes

Une fois ce template validé, les cellules suivantes seront ajoutées progressivement:

1. **Phase 1 - F01-SCANNER**: Depth Anything V2, YOLOv8, SAM
2. **Phase 1 - F02-CORTEX**: Intégration Gemini API
3. **Phase 2 - F03/F04**: Geometry generation & Camera Projection
4. **Phase 3 - F07**: Upscaling (Real-ESRGAN, RIFE)

---

**Version**: 1.0.0  
**Date**: 2026-02-06  
**Auteur**: Vulkan, Maître de la Forge